# Spark Optimization — Combined Notebook

This notebook merges all topics from the original Databricks export (`SparkOptimization.dbc`) into a single sequential notebook.


## Table of Contents
1. [AQE](#aqe)
2. [Scanning Optimization](#scanning-optimization)
3. [Dynamic Partition Pruning](#dynamic-partition-pruning)
4. [Optimize Joins](#optimize-joins)
5. [Broadcast Variable](#broadcast-variable)
6. [DELTA LAKE OPTIMIZATION](#delta-lake-optimization)
7. [SALTING](#salting)
8. [SALTING JOINS (BONUS)](#salting-joins-bonus)
9. [CACHING AND PERSISTENCE](#caching-and-persistence)
10. [SPARK SQL Hints](#spark-sql-hints)


---
## AQE


**Turning OFF the AQE**

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

In [ ]:
spark.conf.get("spark.sql.adaptive.enabled")

Out[8]: 'true'

In [ ]:
from pyspark.sql.functions import * 

In [ ]:
df = spark.read.format("csv")\
            .option("inferSchema",True)\
            .option("header",True)\
            .load("/FileStore/rawdata/BigMart_Sales.csv")

In [ ]:
df.rdd.getNumPartitions()

Out[10]: 1

In [ ]:
display(df)

Item_Identifier | Item_Weight | Item_Fat_Content | Item_Visibility | Item_Type | Item_MRP | Outlet_Identifier | Outlet_Establishment_Year | Outlet_Size | Outlet_Location_Type | Outlet_Type | Item_Outlet_Sales
--- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---
FDA15 | 9.3 | Low Fat | 0.016047301 | Dairy | 249.8092 | OUT049 | 1999 | Medium | Tier 1 | Supermarket Type1 | 3735.138
DRC01 | 5.92 | Regular | 0.019278216 | Soft Drinks | 48.2692 | OUT018 | 2009 | Medium | Tier 3 | Supermarket Type2 | 443.4228
FDN15 | 17.5 | Low Fat | 0.016760075 | Meat | 141.618 | OUT049 | 1999 | Medium | Tier 1 | Supermarket Type1 | 2097.27
FDX07 | 19.2 | Regular | 0.0 | Fruits and Vegetables | 182.095 | OUT010 | 1998 | None | Tier 3 | Grocery Store | 732.38
NCD19 | 8.93 | Low Fat | 0.0 | Household | 53.8614 | OUT013 | 1987 | High | Tier 3 | Supermarket Type1 | 994.7052
FDP36 | 10.395 | Regular | 0.0 | Baking Goods | 51.4008 | OUT018 | 2009 | Medium | Tier 3 | Supermarket Type2 | 556.6088
FD

In [ ]:
df_new = df.groupBy("Item_fat_Content").count()

df_new.display()

Item_fat_Content | count
--- | ---
low fat | 112
Low Fat | 5089
LF | 316
Regular | 2889
reg | 117

---
## Scanning Optimization


**TURN OFF AQE**

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "false")

In [ ]:
# Checking AQE Status
spark.conf.get("spark.sql.adaptive.enabled")

Out[5]: 'false'

In [ ]:
from pyspark.sql.functions import *

**Data Reading**

In [ ]:
df = spark.read.format("csv")\
        .option("inferSchema",True)\
        .option("header",True)\
        .load("/FileStore/rawdata/BigMart_Sales.csv")

In [ ]:
display(df)

Item_Identifier | Item_Weight | Item_Fat_Content | Item_Visibility | Item_Type | Item_MRP | Outlet_Identifier | Outlet_Establishment_Year | Outlet_Size | Outlet_Location_Type | Outlet_Type | Item_Outlet_Sales
--- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---
FDA15 | 9.3 | Low Fat | 0.016047301 | Dairy | 249.8092 | OUT049 | 1999 | Medium | Tier 1 | Supermarket Type1 | 3735.138
DRC01 | 5.92 | Regular | 0.019278216 | Soft Drinks | 48.2692 | OUT018 | 2009 | Medium | Tier 3 | Supermarket Type2 | 443.4228
FDN15 | 17.5 | Low Fat | 0.016760075 | Meat | 141.618 | OUT049 | 1999 | Medium | Tier 1 | Supermarket Type1 | 2097.27
FDX07 | 19.2 | Regular | 0.0 | Fruits and Vegetables | 182.095 | OUT010 | 1998 | None | Tier 3 | Grocery Store | 732.38
NCD19 | 8.93 | Low Fat | 0.0 | Household | 53.8614 | OUT013 | 1987 | High | Tier 3 | Supermarket Type1 | 994.7052
FDP36 | 10.395 | Regular | 0.0 | Baking Goods | 51.4008 | OUT018 | 2009 | Medium | Tier 3 | Supermarket Type2 | 556.6088
FD

**Get No. of Partitions**

In [ ]:
df.rdd.getNumPartitions()

Out[9]: 1

**Changing DEFAULT Partition Size to 128KB**

In [ ]:
# Changing the default partition size to 128KB 

spark.conf.set("spark.sql.files.maxPartitionBytes", 131072)

In [ ]:
df.rdd.getNumPartitions()

Out[13]: 7

**Changing the default partition size to 128MB**

In [ ]:
spark.conf.set("spark.sql.files.maxPartitionBytes", 134217728)

**Repartitioning**

In [ ]:
df = df.repartition(10)

In [ ]:
df.rdd.getNumPartitions()

Out[16]: 10

**Get Partition Info**

In [ ]:
# Function to get the partition id

df = df.withColumn("partition_id",spark_partition_id())

df.display()

Item_Identifier | Item_Weight | Item_Fat_Content | Item_Visibility | Item_Type | Item_MRP | Outlet_Identifier | Outlet_Establishment_Year | Outlet_Size | Outlet_Location_Type | Outlet_Type | Item_Outlet_Sales | partition_id
--- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---
DRG48 | 5.78 | Low Fat | 0.014555066 | Soft Drinks | 145.2102 | OUT046 | 1997 | Small | Tier 1 | Supermarket Type1 | 3062.0142 | 0
FDE24 | 14.85 | Low Fat | 0.09344495 | Baking Goods | 141.0812 | OUT035 | 2004 | Small | Tier 2 | Supermarket Type1 | 1139.8496 | 0
NCP18 | 12.15 | Low Fat | 0.028714747 | Household | 151.9708 | OUT018 | 2009 | Medium | Tier 3 | Supermarket Type2 | 3611.2992 | 0
FDX27 | 20.7 | Regular | 0.114581955 | Dairy | 94.3436 | OUT018 | 2009 | Medium | Tier 3 | Supermarket Type2 | 945.436 | 0
FDU02 | 13.35 | Low Fat | 0.102670882 | Dairy | 228.6352 | OUT049 | 1999 | Medium | Tier 1 | Supermarket Type1 | 3435.528 | 0
FDW57 | 8.31 | Regular | 0.115911972 | Snack Foods | 177.

**Data Writing**

In [ ]:
df.write.format("parquet")\
    .mode("append")\
    .option("path","/FileStore/rawdata/parquetWrite")\
    .save()

**New Data Reading**

In [ ]:
df_new = spark.read.format("parquet")\
              .load("/FileStore/rawdata/parquetWrite")

df_new = df_new.filter(col("Outlet_Location_Type") == 'Tier 1')

In [ ]:
df_new.display()

Item_Identifier | Item_Weight | Item_Fat_Content | Item_Visibility | Item_Type | Item_MRP | Outlet_Identifier | Outlet_Establishment_Year | Outlet_Size | Outlet_Location_Type | Outlet_Type | Item_Outlet_Sales
--- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---
FDU21 | None | Regular | 0.134327613 | Snack Foods | 35.0558 | OUT019 | 1985 | Small | Tier 1 | Grocery Store | 33.9558
FDG08 | None | Regular | 0.289522833 | Fruits and Vegetables | 172.0764 | OUT019 | 1985 | Small | Tier 1 | Grocery Store | 171.7764
FDI60 | 7.22 | Regular | 0.0383808 | Baking Goods | 62.351 | OUT049 | 1999 | Medium | Tier 1 | Supermarket Type1 | 885.514
FDZ12 | 9.17 | Low Fat | 0.102978817 | Baking Goods | 144.947 | OUT046 | 1997 | Small | Tier 1 | Supermarket Type1 | 4294.41
FDZ38 | 17.6 | LF | 0.008001018 | Dairy | 170.4422 | OUT046 | 1997 | Small | Tier 1 | Supermarket Type1 | 2759.0752
DRK13 | 11.8 | Low Fat | 0.115346634 | Soft Drinks | 200.2084 | OUT049 | 1999 | Medium | Tier 1 | Superma

**SCANNING OPTIMIZATION**

In [ ]:
df.write.format("parquet")\
      .mode("append")\
      .partitionBy("Outlet_Location_Type")\
      .option("path","/FileStore/rawdata/parquetWriteOpt")\
      .save()

In [ ]:
df_new = spark.read.format("parquet")\
              .load("/FileStore/rawdata/parquetWriteOpt")

df_new = df_new.filter(col("Outlet_Location_Type") == 'Tier 1')

df_new.display()

Item_Identifier | Item_Weight | Item_Fat_Content | Item_Visibility | Item_Type | Item_MRP | Outlet_Identifier | Outlet_Establishment_Year | Outlet_Size | Outlet_Type | Item_Outlet_Sales | Outlet_Location_Type
--- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---
DRD60 | 15.7 | Low Fat | 0.037289996 | Soft Drinks | 182.7634 | OUT049 | 1999 | Medium | Supermarket Type1 | 3453.5046 | Tier 1
FDY19 | 19.75 | Low Fat | 0.041429246 | Fruits and Vegetables | 117.2466 | OUT049 | 1999 | Medium | Supermarket Type1 | 2239.0854 | Tier 1
NCE06 | None | Low Fat | 0.160178832 | Household | 160.2894 | OUT019 | 1985 | Small | Grocery Store | 323.5788 | Tier 1
NCL31 | None | Low Fat | 0.210596485 | Others | 144.747 | OUT019 | 1985 | Small | Grocery Store | 143.147 | Tier 1
FDZ44 | None | Low Fat | 0.06780958 | Fruits and Vegetables | 118.1808 | OUT019 | 1985 | Small | Grocery Store | 234.3616 | Tier 1
NCT42 | 5.88 | Low Fat | 0.02488732 | Household | 147.5392 | OUT046 | 1997 | Small | Sup

---
## Dynamic Partition Pruning


**Turning OFF AQE and DPP and AutoBroadcast**

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled","true")
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "true")


In [ ]:
df = spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("/FileStore/rawdata/BigMart_Sales.csv")

df = df.limit(8)

In [ ]:
df.display()

Item_Identifier | Item_Weight | Item_Fat_Content | Item_Visibility | Item_Type | Item_MRP | Outlet_Identifier | Outlet_Establishment_Year | Outlet_Size | Outlet_Location_Type | Outlet_Type | Item_Outlet_Sales
--- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---
FDA15 | 9.3 | Low Fat | 0.016047301 | Dairy | 249.8092 | OUT049 | 1999 | Medium | Tier 1 | Supermarket Type1 | 3735.138
DRC01 | 5.92 | Regular | 0.019278216 | Soft Drinks | 48.2692 | OUT018 | 2009 | Medium | Tier 3 | Supermarket Type2 | 443.4228
FDN15 | 17.5 | Low Fat | 0.016760075 | Meat | 141.618 | OUT049 | 1999 | Medium | Tier 1 | Supermarket Type1 | 2097.27
FDX07 | 19.2 | Regular | 0.0 | Fruits and Vegetables | 182.095 | OUT010 | 1998 | None | Tier 3 | Grocery Store | 732.38
NCD19 | 8.93 | Low Fat | 0.0 | Household | 53.8614 | OUT013 | 1987 | High | Tier 3 | Supermarket Type1 | 994.7052
FDP36 | 10.395 | Regular | 0.0 | Baking Goods | 51.4008 | OUT018 | 2009 | Medium | Tier 3 | Supermarket Type2 | 556.6088
FD

**Preparing the Partitioned Data**

In [ ]:
df.write.format("parquet")\
        .mode("append")\
        .partitionBy("Item_identifier")\
        .option("path","/FileStore/rawdata/dpp_partionednew")\
        .save()

**Non Partitioned Data**

In [ ]:
df.write.format("parquet")\
        .mode("append")\
        .option("path","/FileStore/rawdata/dpp_nonpartioned")\
        .save()

**Dataframes**

In [ ]:
df1 = spark.read.format("parquet")\
          .load("/FileStore/rawdata/dpp_partionednew")

In [ ]:
df2 = spark.read.format("parquet")\
          .load("/FileStore/rawdata/dpp_nonpartioned")

**JOINS**

In [ ]:
from pyspark.sql.functions import *

In [ ]:
df_join = df1.join(df2.filter(col("Item_Identifier")=="FDA15"),df1['Item_Identifier']==df2['Item_Identifier'],"inner")

In [ ]:
df_join.display()

Item_Weight | Item_Fat_Content | Item_Visibility | Item_Type | Item_MRP | Outlet_Identifier | Outlet_Establishment_Year | Outlet_Size | Outlet_Location_Type | Outlet_Type | Item_Outlet_Sales | Item_identifier | Item_Identifier | Item_Weight | Item_Fat_Content | Item_Visibility | Item_Type | Item_MRP | Outlet_Identifier | Outlet_Establishment_Year | Outlet_Size | Outlet_Location_Type | Outlet_Type | Item_Outlet_Sales
--- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---
9.3 | Low Fat | 0.016047301 | Dairy | 249.8092 | OUT049 | 1999 | Medium | Tier 1 | Supermarket Type1 | 3735.138 | FDA15 | FDA15 | 9.3 | Low Fat | 0.016047301 | Dairy | 249.8092 | OUT049 | 1999 | Medium | Tier 1 | Supermarket Type1 | 3735.138

In [ ]:
df_join.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   *(2) BroadcastHashJoin [Item_Identifier#118], [Item_Identifier#131], Inner, BuildRight, false, true
   :- *(2) ColumnarToRow
   :  +- FileScan parquet [Item_Weight#107,Item_Fat_Content#108,Item_Visibility#109,Item_Type#110,Item_MRP#111,Outlet_Identifier#112,Outlet_Establishment_Year#113,Outlet_Size#114,Outlet_Location_Type#115,Outlet_Type#116,Item_Outlet_Sales#117,Item_identifier#118] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[dbfs:/FileStore/rawdata/dpp_partionednew], PartitionFilters: [isnotnull(Item_identifier#118), (Item_identifier#118 = FDA15)], PushedFilters: [], ReadSchema: struct<Item_Weight:double,Item_Fat_Content:string,Item_Visibility:double,Item_Type:string,Item_MR...
   +- ShuffleQueryStage 0, Statistics(sizeInBytes=176.0 B, rowCount=1, isRuntime=true)
      +- Exchange SinglePartition, EXECUTOR_BROADCAST, [plan_id=187]
         +- *(1) Filter (isnotnull

---
## Optimize Joins


In [ ]:
from pyspark.sql.functions import *
spark.conf.set("spark.sql.adaptive.enabled", "false")

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "false")

In [ ]:
# Big DataFrame
df_transactions = spark.createDataFrame([
    (1, "US", 100),
    (2, "IN", 200),
    (3, "UK", 150),
    (4, "US", 80),
], ["id", "country_code", "amount"])

# Small DataFrame
df_countries = spark.createDataFrame([
    ("US", "United States"),
    ("IN", "India"),
    ("UK", "United Kingdom"),
], ["country_code", "country_name"])

In [ ]:
df_transactions.display()

id | country_code | amount
--- | --- | ---
1 | US | 100
2 | IN | 200
3 | UK | 150
4 | US | 80

In [ ]:
df_countries.display()

country_code | country_name
--- | ---
US | United States
IN | India
UK | United Kingdom

In [ ]:
df_join_opt = df_transactions.join(broadcast(df_countries),df_transactions['country_code']==df_countries['country_code'],"inner")

In [ ]:
df_join_opt.display()

id | country_code | amount | country_code | country_name
--- | --- | --- | --- | ---
1 | US | 100 | US | United States
2 | IN | 200 | IN | India
3 | UK | 150 | UK | United Kingdom
4 | US | 80 | US | United States

---
## Broadcast Variable


In [ ]:
# Dataframe
df = spark.createDataFrame([
    ("1001",),
    ("1002",),
    ("1004",)
], ["product_id"])

# Lookup dictionary (small)
product_dict = {
    "1001": "iPhone",
    "1002": "Samsung",
    "1003": "Pixel"
}

In [ ]:
df.display()

product_id
---
1001
1002
1004

In [ ]:
# Broadcasting the dictionary variable

broad_vr = spark.sparkContext.broadcast(product_dict)

In [ ]:
# Broadcast Variable Value
broad_vr.value

Out[11]: {'1001': 'iPhone', '1002': 'Samsung', '1003': 'Pixel'}

In [ ]:
# Broadcast Variable Key (1001) Value
broad_vr.value.get('1001')

Out[12]: 'iPhone'

In [ ]:
# Our Function

def mymap(x):

    return broad_vr.value.get(x)

# Converting it into UDF
mymap_udf = udf(mymap)

In [ ]:
# Using udf function and passing "product_id as input to the function"

df_with_names = df.withColumn("product_name", mymap_udf("product_id"))

In [ ]:
df_with_names.display()

product_id | product_name
--- | ---
1001 | iPhone
1002 | Samsung
1004 | None

---
## DELTA LAKE OPTIMIZATION


In [ ]:
# %sql
CREATE SCHEMA ansh_schema

In [ ]:
# %sql
CREATE TABLE ansh_schema.anshtbl
(
  id INT,
  salary INT
)
USING DELTA 
LOCATION '/FileStore/rawdata/deltatbl'

In [ ]:
# %sql
INSERT INTO ansh_schema.anshtbl
VALUES 
(5,100),
(6,100)

num_affected_rows | num_inserted_rows
--- | ---
2 | 2

In [ ]:
# %sql
OPTIMIZE delta.`/FileStore/rawdata/deltatbl` ZORDER BY (id)

path | metrics
--- | ---
dbfs:/FileStore/rawdata/deltatbl | [1, 3, [861, 861, 861.0, 1, 861], [845, 845, 845.0, 3, 2535], 0, ['minCubeSize(107374182400)', [0, 0], [3, 2535], 0, [3, 2535], 1, None], 1, 3, 0, False, 0, 0, 1745096548746, 1745096560267, 8, 1, None, [0, 0], 2, 2, 410]

---
## SALTING


In [ ]:
from pyspark.sql.functions import *

**CREATING A DATAFRAME**

In [ ]:
data = [("A", 100), ("A", 200), ("A", 300), ("B", 400), ("C", 500)]
df = spark.createDataFrame(data, ["user_id", "purchase"])

In [ ]:
df.display()

user_id | purchase
--- | ---
A | 100
A | 200
A | 300
B | 400
C | 500

**ADDING SALT COLUMN**

In [ ]:
df = df.withColumn("salt_column",floor(rand()*3)) # Salt of 3 [0,1,2]

In [ ]:
df.display()

user_id | purchase | salt_column
--- | --- | ---
A | 100 | 0
A | 200 | 0
A | 300 | 1
B | 400 | 2
C | 500 | 2

**Creating Concat Column on original groupBy col and salt_column to create a new groupBy col**

In [ ]:
df = df.withColumn("user_id_salt",concat(col("user_id"),lit("-"),col("salt_column")))

df.display()

user_id | purchase | salt_column | user_id_salt
--- | --- | --- | ---
A | 100 | 0 | A-0
A | 200 | 0 | A-0
A | 300 | 1 | A-1
B | 400 | 2 | B-2
C | 500 | 2 | C-2

**Applying Group By on this new col**

In [ ]:
df = df.groupBy("user_id_salt").agg(sum("purchase"))

df.display()

user_id_salt | sum(purchase)
--- | ---
A-0 | 300
A-1 | 300
B-2 | 400
C-2 | 500

---
## SALTING JOINS (BONUS)


**Create the Dataframes**

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

transactions_data = [("A", 100), ("A", 200), ("A", 300), ("B", 150), ("C", 250)]
transactions_df = spark.createDataFrame(transactions_data, ["user_id", "amount"])

users_data = [("A", "India"), ("B", "USA"), ("C", "UK")]
users_df = spark.createDataFrame(users_data, ["user_id", "country"])


**Salt the transaction_df**

In [ ]:
from pyspark.sql.functions import *

salt_count = 3

# Add a random salt
transactions_salted = transactions_df.withColumn("salt", floor(rand() * salt_count))

# Create salted user_id
transactions_salted = transactions_salted.withColumn("salted_user_id", concat_ws("_", "user_id", "salt"))


In [ ]:
# Displaying transactions_df
transactions_salted.display()

user_id | amount | salt | salted_user_id
--- | --- | --- | ---
A | 100 | 1 | A_1
A | 200 | 0 | A_0
A | 300 | 0 | A_0
B | 150 | 0 | B_0
C | 250 | 2 | C_2

**Adding Salt to users_df (all the values of salt) and then Exploding it**

In [ ]:
from pyspark.sql.functions import explode, array, lit

# Create array of salt values [0,1,2] for each row
users_df = users_df.withColumn("salt", array([lit(i) for i in range(salt_count)]))

# Exploding the values of the array 
users_expanded = users_df.withColumn("salt", explode(col('salt')))

# Create salted user_id
users_expanded = users_expanded.withColumn("salted_user_id", concat_ws("_", "user_id", "salt"))


In [ ]:
# Displaying users_expanded

users_expanded.display()

user_id | country | salt | salted_user_id
--- | --- | --- | ---
A | India | 0 | A_0
A | India | 1 | A_1
A | India | 2 | A_2
B | USA | 0 | B_0
B | USA | 1 | B_1
B | USA | 2 | B_2
C | UK | 0 | C_0
C | UK | 1 | C_1
C | UK | 2 | C_2

**Applying JOIN**

In [ ]:
joined_df = transactions_salted.join(users_expanded, on="salted_user_id", how="inner")

In [ ]:
joined_df.display()

salted_user_id | user_id | amount | salt | user_id | country | salt
--- | --- | --- | --- | --- | --- | ---
A_0 | A | 200 | 0 | A | India | 0
A_0 | A | 300 | 0 | A | India | 0
A_1 | A | 100 | 1 | A | India | 1
B_0 | B | 150 | 0 | B | USA | 0
C_2 | C | 250 | 2 | C | UK | 2

---
## CACHING AND PERSISTENCE


In [ ]:
from pyspark.sql.functions import *

In [ ]:
df = spark.read.format("csv")\
            .option("inferSchema",True)\
            .option("header",True)\
            .load("/FileStore/rawdata/BigMart_Sales.csv")\
            .cache()

In [ ]:
df2 = df.filter(col("Outlet_Location_Type") == 'Tier 1')

In [ ]:
df3 = df.filter(col("Outlet_Location_Type") == 'Tier 2')

In [ ]:
df3.display()

Item_Identifier | Item_Weight | Item_Fat_Content | Item_Visibility | Item_Type | Item_MRP | Outlet_Identifier | Outlet_Establishment_Year | Outlet_Size | Outlet_Location_Type | Outlet_Type | Item_Outlet_Sales
--- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---
FDH17 | 16.2 | Regular | 0.016687114 | Frozen Foods | 96.9726 | OUT045 | 2002 | None | Tier 2 | Supermarket Type1 | 1076.5986
FDU28 | 19.2 | Regular | 0.09444959 | Frozen Foods | 187.8214 | OUT017 | 2007 | None | Tier 2 | Supermarket Type1 | 4710.535
FDU02 | 13.35 | Low Fat | 0.10249212 | Dairy | 230.5352 | OUT035 | 2004 | Small | Tier 2 | Supermarket Type1 | 2748.4224
NCB30 | 14.6 | Low Fat | 0.025698134 | Household | 196.5084 | OUT035 | 2004 | Small | Tier 2 | Supermarket Type1 | 1587.2672
NCD06 | 13.0 | Low Fat | 0.099887103 | Household | 45.906 | OUT017 | 2007 | None | Tier 2 | Supermarket Type1 | 838.908
FDV10 | 7.645 | Regular | 0.066693437 | Snack Foods | 42.3112 | OUT035 | 2004 | Small | Tier 2 | Superma

In [ ]:
df.unpersist()

Out[10]: DataFrame[Item_Identifier: string, Item_Weight: double, Item_Fat_Content: string, Item_Visibility: double, Item_Type: string, Item_MRP: double, Outlet_Identifier: string, Outlet_Establishment_Year: int, Outlet_Size: string, Outlet_Location_Type: string, Outlet_Type: string, Item_Outlet_Sales: double]

In [ ]:
from pyspark.storagelevel import StorageLevel

In [ ]:
df.persist(StorageLevel.MEMORY_ONLY)

---
## SPARK SQL Hints


In [ ]:
from pyspark.sql.functions import *
spark.conf.set("spark.sql.adaptive.enabled", "false")

In [ ]:
# Big DataFrame
df_transactions = spark.createDataFrame([
    (1, "US", 100),
    (2, "IN", 200),
    (3, "UK", 150),
    (4, "US", 80),
], ["id", "country_code", "amount"])

# Small DataFrame
df_countries = spark.createDataFrame([
    ("US", "United States"),
    ("IN", "India"),
    ("UK", "United Kingdom"),
], ["country_code", "country_name"])

In [ ]:
df_transactions.createOrReplaceTempView("transactions")
df_countries.createOrReplaceTempView("countries")

In [ ]:
df_sql_opt = spark.sql(
    '''SELECT /* broadcast(c) */ 
        * 
FROM transactions t 
JOIN countries c 
ON t.country_code = c.country_code''')

In [ ]:
df_sql_opt.display()

id | country_code | amount | country_code | country_name
--- | --- | --- | --- | ---
1 | US | 100 | US | United States
4 | US | 80 | US | United States
2 | IN | 200 | IN | India
3 | UK | 150 | UK | United Kingdom